# bench_grid

> The Phase 8 benchmark grid: main grid and side-sweep cells, a parallel deterministic runner, CSV output, and the pre-registered confirmatory analysis (`PREREG`, `confirm`).

Spec: `specs/multiagent/phase8/` (design-low-level section 4).

In [ ]:
#| default_exp bench_grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import csv
import hashlib
import json
import math
import multiprocessing
import os
from pathlib import Path

import numpy as np

from explRL.bench_metrics import benchMetrics
from explRL.bench_stats import bootCI, cohenDz, holm, signFlipP
from explRL.episodes import runEpisodes
from explRL.harness_runs import HarnessConfig, runHarness

CHANNELS = ('none', 'blackboard', 'messages', 'hierarchical')
SIZES = (1, 2, 4, 8, 16, 32, 64)
PREREG = dict(
    version=1, seeds=list(range(100, 130)), budget='compute', alpha=0.05, nflip=10000, delta=0.5,
    tests=dict(
        P=dict(metric='perAgentSecond', a='blackboard', b='none', sizes=[2, 4, 8, 16, 32, 64], sign=1),
        H1=dict(metric='rmrt', a='blackboard', b='none', sizes=[4, 8, 16, 32, 64], sign=-1),
        H2=dict(metric='diversity', a='blackboard', b='none', sizes=[4, 8, 16, 32, 64], sign=-1)))
SWEEPS = (('scheme', ('team', 'difference'), CHANNELS, 'world'),
          ('lam', (0.0, 0.003, 0.03, 0.1), ('blackboard', 'hierarchical'), 'cfg'),
          ('radius', (0.0, 2.0, 8.0), ('messages',), 'cfg'),
          ('K', (1, 10), CHANNELS, 'cfg'),
          ('shape', ('torus',), CHANNELS, 'world'),
          ('cUCB', (0.25, 0.5, 2.0, 4.0), ('none', 'blackboard'), 'cfg'))
STUDY_B = (('fixed-0.5', 0.0), ('fixed-1', 0.0), ('fixed-2', 0.0), ('fixed-4', 0.0),
           ('strategy', 0.0), ('strategy', 1.0), ('all', 0.0), ('all', 1.0))


def preregHash(prereg=PREREG):
    return hashlib.sha256(json.dumps(prereg, sort_keys=True).encode()).hexdigest()[:12]


def cell(study, factor, level, N, seed, world=None, **cfg):
    return dict(study=study, factor=factor, level=str(level), N=N, seed=seed, runner='harness',
                cfg=HarnessConfig(world=dict(world or {}), **cfg).asDict())


def mainGrid(mode='full'):
    "The B1 cells ('full'), or the same factors on 3 sizes, 2 pilot seeds and budgets / 10 ('quick')."
    sizes, seeds, B, T = ((1, 4, 16), (1000, 1001), 1536.0, 60.0) if mode == 'quick' else (SIZES, PREREG['seeds'], 15360.0, 600.0)
    return [cell('main', 'main', '-', N, s, channel=ch, budget=b, B=B, T=T)
            for ch in CHANNELS for b in ('compute', 'time') for N in sizes for s in seeds]


def sideSweeps(mode='sweeps'):
    "The B9 side-sweep cells and B11 study B sequences ('sweeps'), or one cell per path ('quick')."
    q = mode == 'quick'
    sizes, seeds, B = ((4,), (1000,), 1536.0) if q else ((4, 16, 64), tuple(range(200, 210)), 15360.0)
    cells = []

    def add(factor, level, chans, **kw):
        cells.extend(cell('sweep', factor, level, N, s, channel=ch, budget='compute', B=B, **kw)
                     for ch in (chans[:1] if q else chans) for N in sizes for s in seeds)
    add('baseline', '-', CHANNELS)
    for factor, levels, chans, where in SWEEPS:
        for lv in (levels[:1] if q else levels):
            add(factor, lv, chans, **(dict(world={factor: lv}) if where == 'world' else {factor: lv}))
    cells.extend(cell('sweep', 'circuit', '-', N, s, channel=ch, agent='circuit', budget='time', T=60.0 if q else 600.0)
                 for ch in (('none',) if q else ('none', 'blackboard', 'messages'))
                 for N in ((4,) if q else (4, 16)) for s in seeds)
    for arm, beta in ((('strategy', 1.0),) if q else STUDY_B):
        for N in ((4,) if q else (16, 64)):
            for s in seeds:
                cells.append(dict(study='studyB', factor='studyB', level=f'{arm}|beta={beta}', N=N, seed=s,
                                  runner='episodes', arm=arm, beta=beta, episodes=2 if q else 12, T=60.0 if q else 600.0,
                                  cfg=HarnessConfig(channel='blackboard').asDict()))
    return cells


def runCell(cell):
    "Run one cell in this process; returns its rows (one, or one per episode)."
    if cell['runner'] == 'episodes':
        return runEpisodes(cell['arm'], cell['N'], cell['seed'], cell['episodes'], cell['T'], cell['beta'])
    cfg = HarnessConfig.fromDict(cell['cfg'])
    world, hist, channel, m = runHarness(cfg, cell['N'], cell['seed'])
    return [benchMetrics(cell, world, hist, m, PREREG['delta'])]


def rowKey(r):
    return '|'.join([str(r['study']), str(r['factor']), str(r['level']), str(r['channel']), str(r['budget']),
                     f"{int(r['N']):03d}", f"{int(r['seed']):05d}", f"{int(r.get('episode', 0)):02d}"])


def runGrid(cells, workers=None, run=runCell):
    "Run cells (in `workers` spawned processes); rows sorted by key, so the result does not depend on scheduling."
    workers = workers or min(8, os.cpu_count() or 1)
    if workers == 1:
        out = [run(c) for c in cells]
    else:
        with multiprocessing.get_context('spawn').Pool(workers) as p:
            out = p.map(run, cells, chunksize=4)
    return sorted((r for rows in out for r in rows), key=rowKey)


STR_COLS = {'study', 'factor', 'level', 'channel', 'agent', 'budget', 'arm'}


def _fmt(v):
    if isinstance(v, str):
        return v
    v = float(v)
    return '' if math.isnan(v) else repr(v)


def _parse(k, v):
    if k in STR_COLS or k.startswith('world.'):
        return v
    if v == '':
        return float('nan')
    try:
        return float(v)
    except ValueError:
        return v


def writeCSV(rows, path):
    "Rows to CSV: columns sorted, numbers written as floats with repr, nan and missing as empty."
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    cols = sorted(set().union(*rows)) if rows else []
    with path.open('w', newline='') as f:
        w = csv.writer(f, lineterminator='\n')
        w.writerow(cols)
        for r in rows:
            w.writerow([_fmt(r[c]) if c in r else '' for c in cols])
    return path


def readCSV(path):
    with open(path, newline='') as f:
        return [{k: _parse(k, v) for k, v in r.items()} for r in csv.DictReader(f)]


def confirm(rows, prereg=PREREG, seeds=None):
    "The pre-registered tests: per-seed contrasts averaged over sizes, sign-flip p, Holm, CI, dz and verdict."
    seeds = list(seeds) if seeds is not None else prereg['seeds']
    idx = {(r['channel'], r['budget'], int(r['N']), int(r['seed'])): r for r in rows if r['study'] == prereg.get('study', 'main')}
    res = {}
    for name, t in prereg['tests'].items():
        d, dropped = [], 0
        for s in seeds:
            vals = []
            for N in t['sizes']:
                ra, rb = idx.get((t['a'], prereg['budget'], N, s)), idx.get((t['b'], prereg['budget'], N, s))
                v = (float(ra[t['metric']]) - float(rb[t['metric']])) if ra and rb else float('nan')
                dropped += math.isnan(v)
                if not math.isnan(v):
                    vals.append(v)
            d.append(float(np.mean(vals)) if vals else float('nan'))
        d = np.array(d)
        res[name] = dict(mean=float(np.nanmean(d)) if (~np.isnan(d)).any() else float('nan'), ci=bootCI(d),
                         dz=cohenDz(d), p=signFlipP(d, prereg['nflip']), n=int((~np.isnan(d)).sum()), dropped=dropped)
    for (name, v), pa in zip(res.items(), holm([v['p'] for v in res.values()])):
        v['pHolm'] = float(pa)
        sig = pa < prereg['alpha']
        v['verdict'] = ('supported' if np.sign(v['mean']) == prereg['tests'][name]['sign'] else 'contradicted') if sig \
            else 'not significant'
    return res

## Tests

In [ ]:
import time
import tempfile
from pathlib import Path
import numpy as np
from fastcore.test import test_eq, test_close
from explRL.bench_grid import (PREREG, CHANNELS, SIZES, preregHash, mainGrid, sideSweeps, runCell, runGrid,
                               writeCSV, readCSV, confirm)

# grid sizes and factors (AC-5)
full = mainGrid('full')
test_eq(len(full), 1680)
test_eq({c['cfg']['channel'] for c in full}, set(CHANNELS)); test_eq({c['N'] for c in full}, set(SIZES))
test_eq({c['cfg']['budget'] for c in full}, {'compute', 'time'}); test_eq({c['seed'] for c in full}, set(PREREG['seeds']))
sw = sideSweeps('sweeps')
test_eq(sum(c['runner'] == 'harness' for c in sw), 1350); test_eq(sum(c['runner'] == 'episodes' for c in sw), 160)
assert all(200 <= c['seed'] <= 209 for c in sw)

# the quick grid runs in under 30 s; its CSV round-trips and is identical for 1 and 4 workers
t0 = time.perf_counter(); rows4 = runGrid(mainGrid('quick'), workers=4); el = time.perf_counter() - t0
print(f'quick grid: {len(rows4)} rows in {el:.1f} s'); assert el < 30, el
rows1 = runGrid(mainGrid('quick'), workers=1)
with tempfile.TemporaryDirectory() as td:
    a, b = writeCSV(rows4, Path(td) / 'a.csv'), writeCSV(rows1, Path(td) / 'b.csv')
    test_eq(a.read_bytes(), b.read_bytes())
    back = readCSV(a)
    test_eq(writeCSV(back, Path(td) / 'c.csv').read_bytes(), a.read_bytes())
    test_eq(len(back), 48); test_eq(back[0]['study'], 'main'); assert isinstance(back[0]['perAgentSecond'], float)

# confirm on the quick rows with their pilot seeds (a pipeline check), and on synthetic rows with known effects (AC-7)
q = confirm(back, seeds=(1000, 1001))
test_eq(set(q), {'P', 'H1', 'H2'}); assert q['P']['dropped'] > 0          # sizes 2, 8, 32, 64 are not in the quick grid
rng = np.random.default_rng(0)
syn = []
for s in PREREG['seeds']:
    for N in SIZES:
        base = dict(perAgentSecond=0.3 + rng.normal(0, 0.01), rmrt=20 + rng.normal(0, 1), diversity=0.2 + rng.normal(0, 0.01))
        for ch, shift in (('none', (0, 0, 0)), ('blackboard', (0.1, 5.0, rng.normal(0, 0.01)))):
            syn.append(dict(study='main', channel=ch, budget='compute', N=N, seed=s,
                            perAgentSecond=base['perAgentSecond'] + shift[0], rmrt=base['rmrt'] + shift[1],
                            diversity=base['diversity'] + shift[2]))
c = confirm(syn)
test_eq([c[k]['verdict'] for k in ('P', 'H1', 'H2')], ['supported', 'contradicted', 'not significant'])
assert c['P']['pHolm'] >= c['P']['p'] and c['P']['n'] == 30 and c['P']['dropped'] == 0
test_close(c['P']['mean'], 0.1, eps=1e-9)
test_eq(len(preregHash()), 12)
test_eq(preregHash(), '8a59293847ae')                      # the frozen Phase 8 pre-registration is unchanged
# confirm reads the study from the pre-registration (default 'main')
syn2 = [{**r, 'study': 'other'} for r in syn]
test_eq(confirm(syn2, {**PREREG, 'study': 'other'})['P']['verdict'], 'supported')
assert np.isnan(confirm(syn2)['P']['mean'])

# sideSweeps('quick') runs every runner path: each sweep factor, circuit teams and a study B sequence
qs = runGrid(sideSweeps('quick'), workers=2)
test_eq({r['factor'] for r in qs}, {'baseline', 'scheme', 'lam', 'radius', 'K', 'shape', 'cUCB', 'circuit', 'studyB'})
test_eq(sorted(r['episode'] for r in qs if r['study'] == 'studyB'), [0, 1])
assert [r for r in qs if r['factor'] == 'shape'][0]['world.shape'] == 'torus'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()